In [0]:
%pip install confluent-kafka

In [0]:
import importlib, schema_registry_config, kafka_conn
importlib.reload(schema_registry_config)
importlib.reload(kafka_conn)
from pyspark.sql.functions import col
from pyspark.sql.avro.functions import from_avro


In [0]:
sr_options =  schema_registry_config.get_kafka_registry_config(dbutils)
TOPIC = kafka_conn.TOPIC_REGISTRY
conf = kafka_conn.get_kafka_conf(dbutils=dbutils)
sr_avro_options = schema_registry_config.get_spark_avro_sr_options(dbutils)   # from_avro-shaped
SR_ADDR         = schema_registry_config.get_sr_address(dbutils)    

In [0]:
raw = (spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", conf["bootstrap.servers"])
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("kafka.sasl.jaas.config",
            f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
            f'username="{conf["sasl.username"]}" password="{conf["sasl.password"]}";')
    .option("subscribe", TOPIC)
    .option("startingOffsets", "earliest")
    .load())

parsed = raw.select(
    col("partition"), col("offset"), col("timestamp"),
    from_avro(
        data=col("value"),
        options=sr_avro_options,          # Confluent-prefixed auth keys
        subject=SUBJECT,
        schemaRegistryAddress=SR_ADDR,    # the URL as its own arg (NOT inside options)
    ).alias("event")
).select("partition", "offset", "timestamp", "event.*")


In [0]:
CHK = "s3://retail-data-lake-dev/kafka/_checkpoint/avro_ss_test"
(parsed.writeStream.format("delta").outputMode("append")
    .option("checkpointLocation", CHK)
    .trigger(availableNow=True)
    .toTable("kafka_poc.engagement.avro_ss_test")).awaitTermination()

In [0]:
%sql
select * from kafka_poc.engagement.avro_ss_test